In [ ]:
from dependencies import *

envelopes_log = eelbrain.load.unpickle(PROCESSED_PREDICTOR_DIR / f'~processed_envelopes-log.pickle')
durations = utils.get_durations(envelopes_log)
models = utils.get_models()

In [ ]:
# ESTIMATE TRF

trfs = []
for subject_i in range(len(SUBJECTS)):
    # Standarize subject
    subject = SUBJECTS[subject_i]

    # ————————————————————————————————————————————————————————————————————————————————————————————————
    # Make save directory
    subject_trf_dir = TRF_DIR / subject
    subject_trf_dir.mkdir(exist_ok=True)
    trf_paths = {model: subject_trf_dir / f'{subject} {model}.pickle' for model in models}
    # Skip this subject if all files already exist
    #if all(path.exists() for path in trf_paths.values()): 
        #continue

    # ————————————————————————————————————————————————————————————————————————————————————————————————
    # Load eeg
    eeg_concatenated = subject_eegs[subject]
    print('——————————————————————————————')
    print(f'{subject} eeg extracted')
    
    for model, predictors in models.items():
        # ————————————————————————————————————————————————————————————————————————————————————————————————
        # Skip if the file already exists
        skip = False
        if path.exists():
            try:
                existing_trf = eelbrain.load.unpickle(path)
                if existing_trf is not None:
                    print(f"TRF for {subject} ~ {model} already exists, skipping...\n")
                    skip = True
            except Exception as e:
                print(f"Warning: existing TRF file for {subject} ~ {model} is corrupted or unreadable, will recompute. ({e})")
        if skip:
            continue
        print(f"Estimating: {subject} ~ {model}")
        
        # ————————————————————————————————————————————————————————————————————————————————————————————————
        # Fit the mTRF
        predictors_concatenated = subject_model_predictors[subject][model]
        print('now boosting')
        trf = eelbrain.boosting(eeg_concatenated, predictors_concatenated, -0.100, 1.000, 
                                error='l1', basis=0.050, partitions=5, test=1, selective_stopping=True)
        trfs.append((model, predictors, trf))

        # ————————————————————————————————————————————————————————————————————————————————————————————————
        # Save in directory
        path = trf_paths[model]
        eelbrain.save.pickle(trf, TRF_DIR / path)
        print(f'TRF for {model} complete')